# 🔥 guato_ba — Mapeamento automático de áreas queimadas no Pantanal
### Tutorial passo a passo: do Brazil Data Cube à cicatriz de queimada
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Este notebook mostra, **de ponta a ponta e de forma bem explicada**, como o estudo **guato_ba** detecta áreas queimadas no Pantanal a partir de imagens Sentinel-2. O nome é uma homenagem ao povo **Guató**. A ideia é que você consiga **reproduzir cada passo**:

1. acessar o **Brazil Data Cube (BDC)**;
2. baixar um par de datas com queimada conhecida (tile **21KWB**, **03/08/2024 → 02/09/2024**);
3. visualizar **T1, T2 e a diferença** (índices espectrais);
4. rodar o **modelo (U-Net)** no acumulado anual do tile;
5. montar o **tensor de 5 bandas** por cicatriz (T1, T2, dNDVI, NBR$_{t2}$, dNBR);
6. **filtrar um par de datas** e ver as áreas queimadas;
7. percorrer os polígonos com o **guardrail do Pantanal**.

Cada bloco tem uma explicação curta antes do código. Partes "tensoriais" (manipular raster como matriz) são quebradas em pedacinhos.

</div>

# <span style="color:#cf4c39">🛰️ 1. O que é o Brazil Data Cube (BDC)</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

O **Brazil Data Cube (BDC)** é uma iniciativa do **INPE** que organiza imagens de satélite (Sentinel-2, Landsat, CBERS…) já **corrigidas e alinhadas numa grade regular** — um *data cube*. Em vez de baixar cenas soltas, você consulta um **catálogo STAC** (*SpatioTemporal Asset Catalog*): um padrão que descreve **o que** existe, **onde** e **quando**, e entrega links diretos para as bandas (arquivos **COG** — *Cloud Optimized GeoTIFF*, que permitem ler só um recorte sem baixar tudo).

Usaremos a coleção **`S2_L2A-1`** (Sentinel-2, nível 2A = reflectância de superfície). Cada pixel tem 20 m e vem com uma banda de qualidade **SCL** (classificação de cena: nuvem, sombra, água, vegetação…).

</div>

## <span style="color:#cf4c39">Importando as bibliotecas</span>
<hr style="border:1px solid #e69627;">

In [ ]:
import os, sys
# GroupNorm precisa ser definido ANTES de importar o modelo (senão o checkpoint não carrega):
os.environ["GUATO_NORM"] = "gn"
# fixa os dados do PROJ/GDAL do ambiente geospatial (evita erro de reprojeção):
for k in ("PROJ_DATA","PROJ_LIB"): os.environ[k] = "/opt/conda/envs/geospatial/share/proj"
os.environ["GDAL_DATA"] = "/opt/conda/envs/geospatial/share/gdal"

import numpy as np, rasterio, geopandas as gpd, matplotlib.pyplot as plt
import pystac_client
from rasterio.warp import reproject, Resampling
from affine import Affine

# módulos do próprio estudo (ficam na pasta guato_ba):
GUATO = "/home/jovyan/ID-QUEIMADAS/jupyter/guato_ba"
sys.path.insert(0, GUATO)
import geo_data_science as g
from geo_data_science import (grid_5880, mascara_bioma, geom_por_tile, refl, keep_valido,
                              BAND, SCL_KEEP, WIW_NIR, WIW_SW1, PX_HA, RES, DST)
print("tudo importado ✔")

## <span style="color:#cf4c39">Abrindo o catálogo STAC do BDC</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Abrimos um *client* para o endpoint STAC do BDC. A partir dele faremos buscas por coleção, data e tile.

</div>

In [ ]:
BDC_URL = "https://data.inpe.br/bdc/stac/v1/"
catalogo = pystac_client.Client.open(BDC_URL)
catalogo

# <span style="color:#cf4c39">🔍 2. Um par de datas com queimada conhecida — tile 21KWB</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Vamos trabalhar com o tile **21KWB** (centro-sul do Pantanal) e duas datas em que **sabidamente houve fogo**:

* **T1 = 03/08/2024** (antes/início da cicatriz)
* **T2 = 02/09/2024** (depois — a cicatriz já aparece)

A detecção de queimada é **bitemporal**: comparamos a mesma área em duas datas e procuramos a **queda de vigor da vegetação** (a cicatriz escurece no infravermelho próximo e clareia no SWIR).

</div>

In [ ]:
TILE = "21KWB"; DATA_T1 = "2024-08-03"; DATA_T2 = "2024-09-02"
COLLECTION = ["S2_L2A-1"]

def buscar_cena(tile, dia):
    """Busca no STAC a cena daquele tile naquele dia exato."""
    s = catalogo.search(collections=COLLECTION,
                        query={"bdc:tiles": {"in": [tile]}},
                        datetime=f"{dia}T00:00:00Z/{dia}T23:59:59Z")
    its = list(s.items())
    return its[0] if its else None

item_t1 = buscar_cena(TILE, DATA_T1)
item_t2 = buscar_cena(TILE, DATA_T2)
print("T1:", item_t1.id, "| nuvem %.2f%%" % item_t1.properties["eo:cloud_cover"])
print("T2:", item_t2.id, "| nuvem %.2f%%" % item_t2.properties["eo:cloud_cover"])

## <span style="color:#cf4c39">Definindo a janela de leitura (o "slot" do tile na grade 5880)</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

O estudo trabalha numa grade única em **EPSG:5880, 20 m**. Para ler só o pedaço do tile 21KWB, calculamos a **janela (slot)** dessa grade que cobre a área do tile. É só aritmética de coordenadas: a coluna/linha inicial vem de `(x − x0)/20` e `(y0 − y)/20`.

</div>

In [ ]:
GH, GW, GGT = grid_5880()                 # grade global do estudo
tile_geom = geom_por_tile()[TILE]          # polígono do tile
minx, miny, maxx, maxy = tile_geom.bounds
c0 = int((minx - GGT.c) / RES); r0 = int((GGT.f - maxy) / RES)
sw = int((maxx - minx) / RES); sh = int((maxy - miny) / RES)
slot = Affine(RES, 0, GGT.c + c0*RES, 0, -RES, GGT.f - r0*RES)  # transform da janela
print(f"janela do tile: {sh} linhas x {sw} colunas (20 m) em EPSG:5880")

## <span style="color:#cf4c39">Lendo as bandas — reprojetando o COG para a nossa grade</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Cada banda é um **COG** remoto. `rasterio` + `reproject` leem o recorte e já **reprojetam** para a grade 5880. Usamos 5 bandas: **B04** (vermelho), **B8A** (infravermelho próximo), **B11** (SWIR1), **B12** (SWIR2) e **SCL** (qualidade). Os valores vêm como inteiros (DN); a **reflectância** é `DN/10000`.

</div>

In [ ]:
def ler_banda(item, nome):
    """Lê uma banda do COG e reprojeta para a janela do tile (grade 5880)."""
    destino = np.zeros((sh, sw), np.uint16)
    with rasterio.open(item.assets[nome].href) as src:
        reproject(rasterio.band(src, 1), destino,
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=slot, dst_crs=DST, resampling=Resampling.nearest)
    return destino

bandas = {}
for tag, item in [("t1", item_t1), ("t2", item_t2)]:
    for bn in ("B04", "B8A", "B11", "B12", "SCL"):
        bandas[f"{bn}_{tag}"] = ler_banda(item, bn)
print("bandas lidas:", list(bandas.keys()))

## <span style="color:#cf4c39">Validade do pixel: SCL + WIW (tira nuvem e água)</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Nem todo pixel serve. Mantemos só pixels **SCL ∈ {4,5,7}** (vegetação, solo exposto, sem classificação — i.e. **sem nuvem/sombra**) **e** que **não sejam água** pelo índice **WIW** (*Water In Wetlands*): água tem reflectância baixa no NIR **e** no SWIR1 (`refl(B8A) < 0{,}1804` **e** `refl(B11) < 0{,}1131`). Isso é crucial no Pantanal, que alaga. O modelo **nunca vê** nuvem nem água.

</div>

In [ ]:
def validade(tag):
    b8a = bandas[f"B8A_{tag}"]; b11 = bandas[f"B11_{tag}"]
    b12 = bandas[f"B12_{tag}"]; scl = bandas[f"SCL_{tag}"]
    # keep_valido = (bandas>0) & (SCL em {4,5,7}) & (não-água WIW)
    return keep_valido(b8a, b11, b12, scl)

valido = validade("t1") & validade("t2")   # precisa ser válido NAS DUAS datas
print(f"pixels válidos no tile: {valido.mean()*100:.1f}%")

# <span style="color:#cf4c39">🖼️ 3. Visualizando T1, T2 e a diferença</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Vamos montar **três imagens**, exatamente como na figura do artigo:

* **T1** e **T2** em **falsa-cor SWIR2/NIR8A/Red** (R=B12, G=B8A, B=B04) — nessa composição a **cicatriz fica avermelhada/escura** e se destaca da vegetação (verde).
* **Diferença**: três índices de mudança empilhados como RGB → **dNDVI** no vermelho, **NBR$_{t2}$** no verde e **dNBR** no azul.

Os índices:
* **NDVI** = (NIR−Red)/(NIR+Red) → vigor da vegetação; **dNDVI = NDVI$_{t1}$ − NDVI$_{t2}$** (queda = queimou).
* **NBR** = (NIR−SWIR2)/(NIR+SWIR2) → sensível a queimada; **dNBR = NBR$_{t1}$ − NBR$_{t2}$**.

</div>

## <span style="color:#cf4c39">Esticando o contraste (stretch)</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Os DN ocupam uma faixa estreita no recorte → a imagem sai escura. O **stretch por percentil (3–97%)** espalha os valores para [0,1] e revela a cicatriz. Fazemos por canal.

</div>

In [ ]:
def strm(a, lo, hi):
    return np.clip((a.astype(np.float32) - lo) / (hi - lo + 1e-9), 0, 1)

def falsecor(tag):
    """Composição SWIR2(R)/NIR8A(G)/Red(B) com stretch 3-97%."""
    canais = [bandas[f"B12_{tag}"], bandas[f"B8A_{tag}"], bandas[f"B04_{tag}"]]
    out = []
    for ch in canais:
        v = ch[ch > 0]
        lo, hi = np.percentile(v, 3), np.percentile(v, 97)
        out.append(strm(ch, lo, hi))
    return np.dstack(out)

## <span style="color:#cf4c39">Calculando os índices (como matrizes)</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Cada índice é uma **operação elemento a elemento** entre matrizes de reflectância (mesmo tamanho). `refl()` divide o DN por 10000. O `+1e-6` evita divisão por zero.

</div>

In [ ]:
r1, n1, s1, x1 = refl(bandas["B04_t1"]), refl(bandas["B8A_t1"]), refl(bandas["B11_t1"]), refl(bandas["B12_t1"])
r2, n2, s2, x2 = refl(bandas["B04_t2"]), refl(bandas["B8A_t2"]), refl(bandas["B11_t2"]), refl(bandas["B12_t2"])

ndvi1 = (n1 - r1) / (n1 + r1 + 1e-6); ndvi2 = (n2 - r2) / (n2 + r2 + 1e-6)
nbr1  = (n1 - x1) / (n1 + x1 + 1e-6); nbr2  = (n2 - x2) / (n2 + x2 + 1e-6)
dndvi = ndvi1 - ndvi2       # queda de vigor
dnbr  = nbr1  - nbr2        # queda de NBR (cicatriz)
print("dNBR médio onde válido:", round(float(dnbr[valido].mean()), 3))

In [ ]:
# diferença como RGB: dNDVI (R) / NBR_t2 (G) / dNBR (B)
diff = np.dstack([strm(dndvi, -0.05, 0.30), strm(nbr2, -0.20, 0.40), strm(dnbr, 0.0, 0.35)])

fig, ax = plt.subplots(1, 3, figsize=(17, 6))
ax[0].imshow(falsecor("t1")); ax[0].set_title("T1 — 03/08/2024\nSWIR2/NIR8A/Red")
ax[1].imshow(falsecor("t2")); ax[1].set_title("T2 — 02/09/2024\nSWIR2/NIR8A/Red")
ax[2].imshow(diff);           ax[2].set_title("Diferença\ndNDVI / NBR$_{t2}$ / dNBR")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

# <span style="color:#cf4c39">🧠 4. Rodando o modelo (U-Net) no acumulado 2024 do tile</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Até aqui foi um par de datas. O **produto anual** do guato_ba roda a **U-Net** em **todos os pares consecutivos** de datas limpas do ano (jan→dez) e **une** as detecções. Regras principais:

* **guardrail do Pantanal**: tudo é recortado ao bioma (`mascara_bioma`);
* entrada **sem nuvem/água** (SCL+WIW), inválidos zerados — igual no treino;
* só datas cuja cobertura limpa ≥ 95% do tile;
* limiar **THR = 0,50** e área mínima 24 px (~1 ha).

Usamos o modelo do **split espacial honesto** (`unet_gsplit_norte`). A célula abaixo processa **só o 21KWB** (alguns minutos, precisa de GPU) e salva o resultado em `results/acumulado/`.

</div>

In [ ]:
import accumulate_tile as A      # já importa o modelo com GroupNorm (GUATO_NORM=gn no topo)
CKPT = f"{GUATO}/deeplearning/unet_gsplit_norte/unet_best.pkl"
A.INI, A.FIM = "2024-01-01", "2024-12-31"
resumo = A.run(modelo="unet", tiles=[TILE], ckpt=CKPT, thr=0.50, tag="_tutorial21kwb")
resumo

<div style="text-align: justify; margin-right: 2.5%;">

O resultado vem como **raster** (`acum2024_unet_tutorial21kwb.tif`) e **vetor** (`acum2024_unet_tutorial21kwb.gpkg`), este último com a **data de detecção por polígono**. Vamos carregar o vetor — ele é quem define os **pares de datas** e as **regiões** dos próximos passos.

</div>

In [ ]:
import os
GPKG = f"{g.GUATO}/results/acumulado/acum2024_unet_tutorial21kwb.gpkg"
poligonos = gpd.read_file(GPKG).to_crs(DST)
poligonos["area_ha"] = poligonos.geometry.area / 1e4
poligonos = poligonos.sort_values("area_ha", ascending=False).reset_index(drop=True)
print(f"{len(poligonos)} cicatrizes detectadas | colunas: {list(poligonos.columns)}")
poligonos.head()

# <span style="color:#cf4c39">📦 5. O tensor de 5 bandas por cicatriz</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Para cada cicatriz queremos um **tensor de 5 camadas** (pense num "cubinho" de 5 fatias, todas do mesmo recorte):

| camada | o que é | para que serve |
|---|---|---|
| 1 | **T1** (SWIR2/NIR8A/Red) | antes |
| 2 | **T2** (SWIR2/NIR8A/Red) | depois |
| 3 | **dNDVI** | queda de vigor |
| 4 | **NBR$_{t2}$** | estado pós-fogo |
| 5 | **dNBR** | magnitude da cicatriz |

A função abaixo recebe **duas cenas (t1, t2)** e um **polígono** (a região da cicatriz) e devolve:
* um **tensor numérico** `(5, altura, largura)` — útil se você quiser alimentar um modelo;
* as **5 miniaturas** para visualizar.

Quebrando a parte "tensorial": (a) achamos a **janela** do polígono; (b) lemos as bandas nessa janela; (c) calculamos os índices; (d) **empilhamos** tudo com `np.stack`.

</div>

In [ ]:
def janela_do_poligono(poly, margem_m=1500):
    """(a) bounding box do polígono + margem -> transform e tamanho da janela."""
    mnx, mny, mxx, mxy = poly.bounds
    mnx-=margem_m; mny-=margem_m; mxx+=margem_m; mxy+=margem_m
    w = int((mxx-mnx)/RES); h = int((mxy-mny)/RES)
    return Affine(RES,0,mnx,0,-RES,mxy), h, w, (mnx,mny,mxx,mxy)

def ler_na_janela(item, nome, tr, h, w):
    """(b) lê uma banda do COG já reprojetada para a janela do polígono."""
    o = np.zeros((h, w), np.uint16)
    with rasterio.open(item.assets[nome].href) as s:
        reproject(rasterio.band(s,1), o, src_transform=s.transform, src_crs=s.crs,
                  dst_transform=tr, dst_crs=DST, resampling=Resampling.nearest)
    return o

In [ ]:
def montar_tensor(item_t1, item_t2, poly):
    tr, h, w, bbox = janela_do_poligono(poly)
    rd = lambda it,bn: ler_na_janela(it, bn, tr, h, w)
    # (c) índices a partir da reflectância
    r1,n1,x1 = refl(rd(item_t1,"B04")), refl(rd(item_t1,"B8A")), refl(rd(item_t1,"B12"))
    r2,n2,x2 = refl(rd(item_t2,"B04")), refl(rd(item_t2,"B8A")), refl(rd(item_t2,"B12"))
    ndvi1=(n1-r1)/(n1+r1+1e-6); ndvi2=(n2-r2)/(n2+r2+1e-6)
    nbr1 =(n1-x1)/(n1+x1+1e-6); nbr2 =(n2-x2)/(n2+x2+1e-6)
    dndvi = ndvi1-ndvi2; dnbr = nbr1-nbr2
    # (d) empilha as 5 camadas numéricas (T1 e T2 representados pelo SWIR2, banda-chave de queimada)
    tensor = np.stack([x1, x2, dndvi, nbr2, dnbr], axis=0)   # shape (5, h, w)
    # miniaturas p/ visualizar (T1/T2 em falsa-cor)
    def fc(it):
        ch=[rd(it,"B12"),rd(it,"B8A"),rd(it,"B04")]; o=[]
        for c in ch:
            v=c[c>0]; lo,hi=np.percentile(v,3),np.percentile(v,97); o.append(strm(c,lo,hi))
        return np.dstack(o)
    minis = [("T1", fc(item_t1)), ("T2", fc(item_t2)),
             ("dNDVI", dndvi), ("NBR_t2", nbr2), ("dNBR", dnbr)]
    return tensor, minis, bbox

print("função pronta — tensor (5, h, w) + 5 miniaturas")

## <span style="color:#cf4c39">Exemplo: a maior cicatriz do tile</span>
<hr style="border:1px solid #e69627;">

In [ ]:
poly0 = poligonos.geometry.iloc[0]
tensor, minis, bbox = montar_tensor(item_t1, item_t2, poly0)
print("tensor:", tensor.shape, "| cicatriz:", round(poligonos.area_ha.iloc[0]), "ha")

fig, ax = plt.subplots(1, 5, figsize=(20, 4.2))
for k,(nome,img) in enumerate(minis):
    if img.ndim==3: ax[k].imshow(img)
    else: ax[k].imshow(img, cmap="RdYlGn_r" if nome!="NBR_t2" else "viridis")
    ax[k].set_title(nome); ax[k].axis("off")
plt.tight_layout(); plt.show()

# <span style="color:#cf4c39">🎛️ 6. Filtrando um par de datas e vendo as queimadas</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Agora o pedido-chave: **a pessoa escolhe T1 e T2** (entre as datas limpas disponíveis no tile) e o notebook mostra **onde queimou** naquele par. Fazemos com `ipywidgets`: dois menus suspensos + um gráfico que recalcula o **dNBR** e aplica um **limiar** (a mesma lógica de mudança temporal).

</div>

In [ ]:
# datas limpas (<20% nuvem) disponíveis no tile em 2024
itens = g.buscar_itens(TILE, "2024-05-01", "2024-11-30", 20.0)
por_data = {it.properties["datetime"][:10]: it for it in itens}
datas = sorted(por_data)
print(len(datas), "datas disponíveis:", datas)

In [ ]:
import ipywidgets as W
from IPython.display import display

def mostrar_par(t1, t2, limiar=0.15):
    """Lê o par, calcula dNBR no tile e mostra a máscara de queimada (dNBR > limiar, só em pixels válidos)."""
    it1, it2 = por_data[t1], por_data[t2]
    n1 = refl(ler_banda(it1,"B8A")); x1 = refl(ler_banda(it1,"B12"))
    n2 = refl(ler_banda(it2,"B8A")); x2 = refl(ler_banda(it2,"B12"))
    vld = validade_par(it1, it2)
    dnbr = (n1-x1)/(n1+x1+1e-6) - (n2-x2)/(n2+x2+1e-6)
    queimada = (dnbr > limiar) & vld
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(dnbr, cmap="RdYlGn_r", vmin=-0.2, vmax=0.5); ax[0].set_title(f"dNBR {t1} → {t2}")
    ax[1].imshow(queimada, cmap="gray"); ax[1].set_title(f"queimada (dNBR>{limiar}) = {queimada.sum()*PX_HA:,.0f} ha")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

def validade_par(it1, it2):
    v1 = keep_valido(ler_banda(it1,"B8A"),ler_banda(it1,"B11"),ler_banda(it1,"B12"),ler_banda(it1,"SCL"))
    v2 = keep_valido(ler_banda(it2,"B8A"),ler_banda(it2,"B11"),ler_banda(it2,"B12"),ler_banda(it2,"SCL"))
    return v1 & v2

In [ ]:
# menus interativos: escolha T1 e T2 e veja a queimada do par
W.interact(mostrar_par,
           t1=W.Dropdown(options=datas, value="2024-08-03", description="T1"),
           t2=W.Dropdown(options=datas, value="2024-09-02", description="T2"),
           limiar=W.FloatSlider(min=0.05, max=0.4, step=0.01, value=0.15, description="limiar dNBR"));

<div style="text-align: justify; margin-right: 2.5%;">

💡 **Quem define os pares que fazem sentido é o vetor de polígonos** (passo 4): cada cicatriz tem a sua `data_deteccao`, indicando entre quais datas ela surgiu. Você pode usar essa coluna para pré-selecionar os menus acima em vez de testar no escuro.

</div>

# <span style="color:#cf4c39">🔁 7. Percorrendo as cicatrizes com o guardrail do Pantanal</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Por fim, o **loop** que percorre os polígonos detectados e, para cada um, **confere se está dentro do bioma Pantanal** (a regra inegociável do estudo — nada fora do bioma conta) e monta o tensor de 5 bandas daquela cicatriz. É a mesma lógica usada no produto final.

</div>

In [ ]:
_, BIOMA_GEOM = g._bioma_geom()     # polígono do bioma Pantanal (união dos tiles)

for i, row in poligonos.head(3).iterrows():
    poly = row.geometry
    dentro = poly.representative_point().within(BIOMA_GEOM)   # guardrail
    print(f"cicatriz #{i+1}: {row.area_ha:,.0f} ha | dentro do Pantanal? {dentro}")
    if not dentro:
        print("   -> fora do bioma: IGNORADA (guardrail)"); continue
    tensor, minis, _ = montar_tensor(item_t1, item_t2, poly)
    fig, ax = plt.subplots(1, 5, figsize=(18, 3.6))
    for k,(nome,img) in enumerate(minis):
        ax[k].imshow(img if img.ndim==3 else img, cmap=None if img.ndim==3 else "RdYlGn_r")
        ax[k].set_title(f"#{i+1} {nome}", fontsize=8); ax[k].axis("off")
    plt.tight_layout(); plt.show()

# <span style="color:#cf4c39">✅ Resumo</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Você percorreu o ciclo completo do **guato_ba**:

1. **BDC/STAC** — acesso às imagens Sentinel-2;
2. **par 03/08 → 02/09 (21KWB)** — leitura de bandas + validade (SCL+WIW);
3. **T1 / T2 / diferença** — falsa-cor e índices (dNDVI, NBR, dNBR);
4. **modelo U-Net** no acumulado anual do tile → vetor de cicatrizes;
5. **tensor de 5 bandas** por cicatriz;
6. **filtro interativo** de par de datas → máscara de queimada;
7. **loop com o guardrail do Pantanal**.

Os produtos anuais completos (2024, 2025, 2026) estão em `results/acumulado/` (`acum<ano>_unet_gsplit.{tif,gpkg}`). Para trocar de tile, mude `TILE`; para trocar de modelo, aponte `CKPT` para outro checkpoint.

</div>